# CB2KHY1C2G9PT

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from foodcast.tools.filter import FilterDF as fdf
from foodcast.tools.benchmarks import ParetoAnalysis as pa
from foodcast.tools.benchmarks import AccuracyCalculation as ac
from foodcast.tools.integrity_fixes import DataFixer as fix, DataExporter as exporter
from foodcast.tools.coverage_functions import plot_time_series
from foodcast.tools.labeling_functions import plot_dish_time_series, fully_relabel_and_consolidate

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/3_data_parquet_relabeled/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/3_data_parquet_relabeled/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/3_data_parquet_relabeled/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

loc_id = 'CB2KHY1C2G9PT'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
plot_dish_time_series(df_uncleaned.query('item_modifications.str.contains("Vegan")'), loc_id, before_after_details_true)

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.query('item_name.str.contains("Impossible")')['item_quantity'].sum()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Impossible")')['item_quantity'].sum()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='W')
plt.show()

In [ ]:
plot_time_series_subset(loc_id, 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='D')
plt.show()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned.query('item_name.str.contains("Impossible")'), 
                 before_after_details_true, 
                 freq='W')
plt.show()

In [ ]:
plot_time_series_subset(loc_id, 
                 df_uncleaned.query('item_name.str.contains("Impossible")'), 
                 before_after_details_true, 
                 freq='D')
plt.show()

Time Differences

In [ ]:
sales_and_menu_data[loc_id].index

In [ ]:
time_differences_details[loc_id]

In [ ]:
# Item names to consolidate
name_changes = {"Impossible Patty Melt" : ["Impossible Melt"],
              "Gold Standard - Impossible" : ["Gold Standard Impossible", "Imp Breakfast"], 
              "Gold Standard - Bacon" : ["Gold Standard - Bacon!", "Gold Standard Breakfast Sandwich"],
              "Gold Standard - Bacon & Kale" : ["Gold Standard - Bacon/Kale", "Gold Standard - Bacon + Kale", "Both - Gold Standard", "Gold Standard - Bacon Ü•Ì"],  
              "Telway Burger" : ["Telway"], 
              "The Alternative" : ["Alternative"],
              "Beyond Burger" : ["Beyond Burg"],
              "Bulldog Butty" : ["Butty"],
              "Four Guys Burger" : ["Four Guys", "4 Guys", "Guys"],
              "Iced Coffee" : ["Cold Brew Coffee", "Cold Brew Coffee ‚Òïô∏È"],
              "Tea" : ["Ice T"],
              "Mexican Coca-Cola": ["Mexi-Coke"],
              "Miss Vickie" : ["Miss Vickie‚Äôs"]}

# Swap the keys and values
name_changes_dict = {variant: canonical for canonical, variants in name_changes.items() for variant in variants}

# Item names to swap based on modications
modification_name_changes = [('Gold Standard - Impossible', 'Add Bacon|Extra Bacon|Regular Bacon|Beef', 'Gold Standard - Bacon/Beef & Impossible'),
                             ('Beyond Burger', 'Add Bacon|Beef', 'Bacon/Beef Beyond Burger'),
                             ('Beyond Burger', 'No Cheese|No Chz|No C|No Cuz|Vegan', 'Vegan Beyond Burger'),
                             ('Impossible Patty Melt', 'Add Bacon|Beef|Meat', 'Bacon/Beef Impossible Patty Melt'),
                             ('Impossible Patty Melt', 'No Cheese|No Chz|No C|No Cuz|Vegan', 'Vegan Impossible Patty Melt'),
                             ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'Sub Impossible', 'Gold Standard - Impossible'),
                             ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'Add Impossible', 'Gold Standard - Bacon & Impossible'),
                             ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'No Bacon', 'Gold Standard - Kale'),
                             ('Gold Standard - Kale', 'Bacon And Kale', 'Gold Standard - Bacon & Kale'),
                             ('Gold Standard - Kale', 'Impossible', 'Gold Standard - Impossible'),
                             ('Gold Standard Sandwich', 'Bacon|Impossible', 'Gold Standard - Bacon & Impossible'),
                             ('Gold Standard Sandwich', 'Impossible', 'Gold Standard - Impossible'),
                             ('Gold Standard Sandwich', 'Vegan', 'Vegan Gold Standard'),
                             ('Gold Standard Sandwich', '', 'Gold Standard - Kale'),
                             ('Scrambled Eggs', 'Bacon|Sausage|Trout', 'Scrambled Eggs With Meat'),
                             ('Eggs Federal', 'Bacon|Sausage|Trout', 'Two Eggs Any Style With Meat'),
                             ('Two Eggs Any Style', 'Bacon|Sausage|Trout', 'Two Eggs Any Style With Meat')]

# Turn into dataframe for viewing
modification_name_changes_df = pd.DataFrame(data = modification_name_changes, columns = ['name', 'modification', 'new_name'])

alcoholic_drinks = []

non_alcoholic_drinks = [
    "Juice",
    "Coca-Cola",
    "Coconut Water",
    "Mexican Coca-Cola",
    "La Croix",
    "Water",
    "Ginger Beer",
    "Coffee",
    "Cheerwine",
    "Iced Coffee",
    "Mata Mate",
    "Mate Libre",
    "Zamalek",
    "Crodino",
    "Lemonade",
    "Club Mate",
    "Tea",
    "Cream Soda",
    "Egg Nog",
    "Mate"
]

merch = [
    "Egift Card",
]

rare = [
    "Blah",
    "Laura Palmer",
    "Ketchup Pack",
    "Donut",
    "Coney Island Dog"
]

unknown = [
    "Og FaveÜç¶",
    "Mom Jeans",
    "Cool",
    "Hi"
]

vegetarian = [
    'Gold Standard - Kale', 
    'Gold Standard - Impossible', 
    'Impossible Patty Melt', 
    'Beyond Burger', 
    'Cole Slaw', 
    'Miss Vickie', 
    'Ice Cream Sandwich'
]

vegan = [
    "The Alternative",
    "Vegan Beyond Burger",
    "Vegan Impossible Patty Melt"
]

items_to_remove = []

In [ ]:
# Bacon or Bacon & Kale
# No Bacon -> vegetarian
# No Cheese -> carnist
# No Cheese, No Bacon, No Aioli -> vegan

# Kale
# Add Impossible -> Impossible
# 

# Impossible
# Add Bacon -> carnist
#

# Bacon & Impossible

animal_product_conditions = ['item_modifications.str.contains("Bacon")',
                             'item_modifications.str.contains("Cheese")',
                             'item_modifications.str.contains("Impossible")',
                             'item_modifications.str.contains("Aioli")',
                             'item_modifications.str.contains("Egg")',
                             'item_modifications.str.contains("Cheddar")',
                             'item_modifications.str.contains("Sub")'
                             ]
print(df.query('item_name.str.contains("Beyond") and (' + ' or '.join(animal_product_conditions) + ')')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
print(df.query('item_name.isin(["Scrambled Eggs", "Eggs Federal", "Two Eggs Any Style"])')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
print(df.query('item_name == "Gold Standard Sandwich"')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
print(df['item_name'].value_counts().to_string())

In [ ]:
df_cleaned = (df
              .assign(item_name = lambda df: df['item_name']
                      .str.strip('123456789./\\ ')  # Clean up item names
                      .replace(name_changes_dict),    # Consolidate dishes
                      )
              .assign(item_name = lambda df: np.select(condlist = [df['item_name'].eq(name) &  
                                                                   df['item_modifications'].str.contains(modification) for name, modification, _ in modification_name_changes],
                                                       choicelist = modification_name_changes_df['new_name'].tolist(),
                                                       default = df['item_name']),
                      dish_category = lambda df: 
                              df['dish_category']
                              .mask(df['item_name'].isin(non_alcoholic_drinks), 'Drink')
                              .mask(df['item_name'].isin(unknown), 'Unknown'),
                      vegetarian = lambda df: df['item_name'].isin(vegetarian + vegan + non_alcoholic_drinks),
                      vegan = lambda df: df['item_name'].isin(vegan + non_alcoholic_drinks)
                      )
              #.drop('unique_id', axis=1)
             )

df = df_cleaned
food_df = df.query('~dish_category.isin(["Alcohol", "Drink", "Merch", "Rare"])')

In [ ]:
print(df['item_name'].value_counts().to_string())

In [ ]:
promo_datetime = before_after_details.loc[loc_id,'cross_over_date'].tz_convert('UTC')

In [ ]:
dish_df['item_price'].min()

In [ ]:
df.query('item_name == "Gold Standard - Bacon & Impossible"')['item_price'].describe()

In [ ]:
df.query('item_name == "Two Eggs Any Style With Meat"')['item_price'].describe()

In [ ]:
df.query('item_name == "Kale Caesar"')['item_price'].describe()

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

top_n = 30

unique_dishes = (df
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c')
                 [:top_n]
                 .index[::-1]
                 )

legend_handles = []

for dish in unique_dishes:
    
    dish_df = food_df.query('item_name == @dish')
    
    vmin = dish_df['unit_price'].min()
    vmax = dish_df['unit_price'].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.ScalarMappable(norm=norm, cmap='magma')
    
    weekly_quantities = (dish_df
                         .resample('W')
                         .agg({'item_quantity': 'sum', 'unit_price': 'mean'})
                         .query('0 < item_quantity')
                         .assign(week = lambda df: df.index.tz_localize(None).to_period('W'))
                         .set_index('week')
                         )
    
    # For every active week
    for week, row  in weekly_quantities.iterrows():
        
        weekly_quantity = row['item_quantity']
        dot_size = 3#weekly_quantity/10 + 3
        weekly_price = row['unit_price']
        color = cmap.to_rgba(weekly_price)

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors=color, lw=dot_size, label=loc_id)
        
    ax.text(x=food_df.index[-1] + pd.DateOffset(100), y=dish, s=f'${vmin/100:.2f}-${vmax/100:.2f}', verticalalignment='center', horizontalalignment='left', fontsize='x-small', color='gray')
    
    # Create a custom legend entry for this dish
    #color_patch_min = mpatches.Patch(color=cmap.to_rgba(vmin), label=f'{dish} Min: ${vmin/100:.2f}')
    #color_patch_max = mpatches.Patch(color=cmap.to_rgba(vmax), label=f'{dish} Max: ${vmax/100:.2f}')
    #legend_handles.extend([color_patch_min, color_patch_max])

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title(f'Weekly Sales of Top {top_n} Dishes for {loc_id}')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')
#ax.legend(handles=legend_handles, title="Price Range per Dish", fontsize='small', loc='upper left', bbox_to_anchor=(1, 1))

# Figure
introduction_fig.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

top_n = 30

unique_dishes = (food_df
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c')
                 [:top_n]
                 .index[::-1]
                 )

legend_handles = []

for dish in unique_dishes:
    
    dish_df = food_df.query('item_name == @dish')
    
    vmin = dish_df['unit_price'].min()
    vmax = dish_df['unit_price'].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.ScalarMappable(norm=norm, cmap='magma')
    
    weekly_quantities = (dish_df
                         .resample('W')
                         .agg({'item_quantity': 'sum', 'unit_price': 'mean'})
                         .query('0 < item_quantity')
                         .assign(week = lambda df: df.index.tz_localize(None).to_period('W'))
                         .set_index('week')
                         )
    
    # For every active week
    for week, row  in weekly_quantities.iterrows():
        
        weekly_quantity = row['item_quantity']
        dot_size = 3#weekly_quantity/10 + 3
        weekly_price = row['unit_price']
        color = cmap.to_rgba(weekly_price)

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors=color, lw=dot_size, label=loc_id)
        
    ax.text(x=food_df.index[-1] + pd.DateOffset(100), y=dish, s=f'${vmin/100:.2f}-${vmax/100:.2f}', verticalalignment='center', horizontalalignment='left', fontsize='x-small', color='gray')
    
    # Create a custom legend entry for this dish
    #color_patch_min = mpatches.Patch(color=cmap.to_rgba(vmin), label=f'{dish} Min: ${vmin/100:.2f}')
    #color_patch_max = mpatches.Patch(color=cmap.to_rgba(vmax), label=f'{dish} Max: ${vmax/100:.2f}')
    #legend_handles.extend([color_patch_min, color_patch_max])

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title(f'Weekly Sales of Top {top_n} Dishes for {loc_id}')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')
#ax.legend(handles=legend_handles, title="Price Range per Dish", fontsize='small', loc='upper left', bbox_to_anchor=(1, 1))

# Figure
introduction_fig.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

top_n = 30

unique_dishes = (food_df
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c')
                 [:top_n]
                 .index[::-1]
                 )

legend_handles = []

for dish in unique_dishes:
    
    dish_df = food_df.query('item_name == @dish')
    
    vmin = dish_df['unit_price'].min()
    vmax = dish_df['unit_price'].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.ScalarMappable(norm=norm, cmap='magma')
    
    weekly_quantities = (dish_df
                         .resample('W')
                         .agg({'item_quantity': 'sum', 'unit_price': 'mean'})
                         .query('0 < item_quantity')
                         .assign(week = lambda df: df.index.tz_localize(None).to_period('W'))
                         .set_index('week')
                         )
    
    # For every active week
    for week, row  in weekly_quantities.iterrows():
        
        weekly_quantity = row['item_quantity']
        dot_size = weekly_quantity/10 + 3
        weekly_price = row['unit_price']
        color = cmap.to_rgba(weekly_price)

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors=color, lw=dot_size, label=loc_id)
        
    ax.text(x=food_df.index[-1] + pd.DateOffset(100), y=dish, s=f'${vmin/100:.2f}-${vmax/100:.2f}', verticalalignment='center', horizontalalignment='left', fontsize='x-small', color='gray')
    
    # Create a custom legend entry for this dish
    #color_patch_min = mpatches.Patch(color=cmap.to_rgba(vmin), label=f'{dish} Min: ${vmin/100:.2f}')
    #color_patch_max = mpatches.Patch(color=cmap.to_rgba(vmax), label=f'{dish} Max: ${vmax/100:.2f}')
    #legend_handles.extend([color_patch_min, color_patch_max])

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title(f'Weekly Sales of Top {top_n} Dishes for {loc_id}')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')
#ax.legend(handles=legend_handles, title="Price Range per Dish", fontsize='small', loc='upper left', bbox_to_anchor=(1, 1))

# Figure
introduction_fig.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [ ]:
weekly_quantities

In [ ]:
df